# Alphamind新手入门之九：股票选股回测

> 在本例中，使用公式选股并进行回测

In [ ]:
import asyncio


## 1. 参数设置
----------------

* 调仓频率：每周
* 股票池  ：中证800
* 起始日  ：2017-05-31
* 结束日  ：2019-05-31
* 业绩基准：沪深300

In [ ]:
import datetime as dt
import numpy as np


## 2. 选股公式
-----------------------

* 在每个调仓日获取`EMA5D`因子排序，最高的3支股票。

In [ ]:
import pyarrow as pa


## 3. 计算过程
----------------

* 获取因子值以及行业信息

In [ ]:
import pyarrow.compute as pc


* 因子选股并且进行获取相关股票每日收益

In [ ]:
rng = np.random.default_rng(42)
n_dates = 6


* 计算每日调仓结果以及收益

In [ ]:
n_assets = 12


## 4. 表现展示
-------------------

### 4.1 日收益与回撤

In [ ]:
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]


In [ ]:
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets


* 累积收益曲线 + 回撤曲线

In [ ]:
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})


* 最大回撤、年化收益

In [ ]:
from alphamind.data.transformer import Transformer


### 4.2 超额收益以及alpha

In [ ]:
from alphamind.execution.naiveexecutor import NaiveExecutor
from alphamind.formula import CSRank


In [ ]:
from alphamind.portfolio.rankbuilder import rank_build


In [ ]:
scores = await asyncio.to_thread(
    Transformer({"score": CSRank("factor")}).transform, "code", panel
)


### 4.3 月度收益

In [ ]:
ranked_panel = panel.append_column("score", scores["score"])
executor = NaiveExecutor()


### 4.4 行业分布

In [ ]:
results = []


### 4.5 换手率

* 月换手直方图

In [ ]:
for trade_date in dates:
    section = ranked_panel.filter(pc.equal(ranked_panel["trade_date"], trade_date))
    weights = rank_build(
        section["score"].to_numpy(zero_copy_only=False), use_rank=3
    ).flatten().astype(float) / 3.0
    target = pa.table({"code": section["code"], "weight": weights})
    turn_over, executed = executor.execute(target)
    executor.set_current(executed)
    result = float(weights @ section["dx"].to_numpy(zero_copy_only=False))
    results.append(pa.table({
        "trade_date": [trade_date], "return": [result], "turn_over": [turn_over]
    }))


In [ ]:
backtest = pa.concat_tables(results)
assert backtest.num_rows == n_dates


In [ ]:
backtest
